# 02_02 — Preparación cartográfica SER

Este notebook prepara las capas cartográficas utilizadas para representar el ámbito del Servicio de Estacionamiento Regulado (SER) de Madrid. La unidad espacial cambia según la fuente: polígono del límite SER, polígonos de barrios SER, líneas de bandas de aparcamiento y polígonos de superficie vial.

La granularidad temporal es estática o corresponde a la versión actual descargada de cada fuente. El notebook no construye dificultad SER, agregaciones horarias, paneles, joins finales, métricas proxy ni modelos.

Las cuatro fuentes se procesan individualmente para obtener capas geoespaciales limpias, reproducibles y adecuadas para las validaciones espaciales y visualizaciones posteriores. El callejero se utilizará como fondo vial neutro para representar las bandas reguladas y distinguir, cuando la geometría lo permita, los lados de estacionamiento de una misma calle. Su utilidad para el mapa final queda condicionada a los resultados de calidad y cobertura.

El flujo comienza con la configuración y la inspección estructural de los datos raw. A continuación se limpian individualmente el límite SER, los barrios SER y las bandas de aparcamiento. La preparación específica del callejero se abordará después de cerrar estas tres capas Geoportal.

## 0. Configuración inicial

La raíz del repositorio se detecta mediante la existencia de `data_catalog.csv`. Todas las rutas se gestionan de forma relativa al repositorio para evitar dependencias del entorno de ejecución.

Las operaciones espaciales se realizan en ETRS89 / UTM zona 30N, EPSG:25830. Este sistema de referencia permite expresar áreas, longitudes y tolerancias espaciales en metros.

Las fuentes de entrada se localizan bajo `data/raw/cartografia/`. Las salidas limpias se escriben en `data/interim/cartografia/` únicamente después de superar el diagnóstico, las validaciones de cada fuente y el control de equivalencia con las salidas de referencia.

In [1]:
from __future__ import annotations

import tempfile
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv al recorrer la ruta actual y sus padres. "
        f"Ruta inicial: {current}"
    )


ROOT = find_repo_root()
CATALOG_PATH = ROOT / "data_catalog.csv"
TARGET_CRS = "EPSG:25830"

TARGET_DATASET_IDS = [
    "ser_geoportal_limite_ser",
    "ser_geoportal_barrios_ser",
    "ser_geoportal_bandas_aparcamiento",
    "callejero_viales_vigentes",
]

CARTOGRAPHY_RAW_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser.geojson"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser.geojson"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_bandas_aparcamiento/"
        / "SHP_ZIP.zip"
    ),
    "callejero_viales_vigentes": (
        ROOT
        / "data/raw/cartografia/callejero_viales_vigentes/"
        / "contexto_callejero_viales_vigentes__actual.zip"
    ),
}

CANDIDATE_INTERIM_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser_clean.parquet"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser_clean.parquet"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_bandas_aparcamiento/"
        / "ser_geoportal_bandas_aparcamiento_clean.parquet"
    ),
    "callejero_viales_vigentes": (
        ROOT
        / "data/interim/cartografia/callejero_viales_vigentes/"
        / "callejero_viales_vigentes_clean.parquet"
    ),
}

LEGACY_INTERIM_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/interim/ser/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser_clean.parquet"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/interim/ser/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser_clean.parquet"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/interim/ser/ser_geoportal_bandas_aparcamiento/"
        / "ser_geoportal_bandas_aparcamiento_clean.parquet"
    ),
}

expected_keys = set(TARGET_DATASET_IDS)
if set(CARTOGRAPHY_RAW_PATHS) != expected_keys:
    raise ValueError(
        "Las claves de CARTOGRAPHY_RAW_PATHS no coinciden con TARGET_DATASET_IDS. "
        f"Observado: {sorted(CARTOGRAPHY_RAW_PATHS)}; esperado: {TARGET_DATASET_IDS}"
    )
if set(CANDIDATE_INTERIM_PATHS) != expected_keys:
    raise ValueError(
        "Las claves de CANDIDATE_INTERIM_PATHS no coinciden con TARGET_DATASET_IDS. "
        f"Observado: {sorted(CANDIDATE_INTERIM_PATHS)}; esperado: {TARGET_DATASET_IDS}"
    )

expected_legacy_keys = {
    "ser_geoportal_limite_ser",
    "ser_geoportal_barrios_ser",
    "ser_geoportal_bandas_aparcamiento",
}
if set(LEGACY_INTERIM_PATHS) != expected_legacy_keys:
    raise ValueError(
        "Las claves de LEGACY_INTERIM_PATHS no coinciden con las tres fuentes Geoportal. "
        f"Observado: {sorted(LEGACY_INTERIM_PATHS)}; esperado: {sorted(expected_legacy_keys)}"
    )

print(f"ROOT: {ROOT}")
print(f"GeoPandas: {gpd.__version__}")
print(f"CRS objetivo: {TARGET_CRS}")

ROOT: /Users/hugo/TFM_parking_madrid
GeoPandas: 1.1.3
CRS objetivo: EPSG:25830


Si no se detecta la raíz del repositorio o GeoPandas no puede cargarse, el notebook no debe continuar. El CRS objetivo permite realizar de forma coherente las operaciones métricas necesarias para la preparación cartográfica.

Las rutas se definen explícitamente para garantizar la reproducibilidad y la trazabilidad de cada fuente y de sus futuras salidas limpias.

## 1. Fuentes de entrada y salidas previstas

El notebook mantiene un identificador estable para cada una de las cuatro fuentes cartográficas y define de forma explícita sus rutas raw y sus salidas interim previstas.

La separación entre datos raw e interim evita modificar las fuentes originales y permite que cada capa limpia se genere únicamente después de superar sus comprobaciones de estructura, calidad geométrica y utilidad para el TFM.

In [2]:
def _relative_to_root(path: Path) -> str:
    try:
        return path.relative_to(ROOT).as_posix()
    except ValueError:
        return str(path)


def _route_error(dataset_id: str, path: Path, observed: Any, expected: str) -> ValueError:
    return ValueError(
        f"Dataset: {dataset_id}; ruta: {_relative_to_root(path)}; "
        f"valor observado: {observed}; condición esperada: {expected}"
    )


route_rows = []
for dataset_id in TARGET_DATASET_IDS:
    raw_path = CARTOGRAPHY_RAW_PATHS[dataset_id]
    candidate_path = CANDIDATE_INTERIM_PATHS[dataset_id]
    legacy_path = LEGACY_INTERIM_PATHS.get(dataset_id)

    if raw_path.suffix.lower() == ".zip" and raw_path.exists():
        with zipfile.ZipFile(raw_path) as archive:
            internal_files = [name for name in archive.namelist() if not name.endswith("/")]
        n_archivos_internos = len(internal_files)
        n_shapefiles_en_zip = sum(name.lower().endswith(".shp") for name in internal_files)
    elif raw_path.suffix.lower() in {".geojson", ".json"}:
        n_archivos_internos = 1
        n_shapefiles_en_zip = pd.NA
    else:
        n_archivos_internos = pd.NA
        n_shapefiles_en_zip = pd.NA

    route_rows.append(
        {
            "dataset_id": dataset_id,
            "archivo_raw_candidato": _relative_to_root(raw_path),
            "raw_existe": raw_path.exists(),
            "n_archivos_internos": n_archivos_internos,
            "n_shapefiles_en_zip": n_shapefiles_en_zip,
            "archivo_interim_candidato": _relative_to_root(candidate_path),
            "interim_candidato_existe": candidate_path.exists(),
            "archivo_interim_legacy": pd.NA if legacy_path is None else _relative_to_root(legacy_path),
            "interim_legacy_existe": pd.NA if legacy_path is None else legacy_path.exists(),
        }
    )

route_contract = pd.DataFrame(route_rows)

for dataset_id, raw_path in CARTOGRAPHY_RAW_PATHS.items():
    if not raw_path.exists():
        raise _route_error(dataset_id, raw_path, raw_path.exists(), "el raw candidato debe existir")

raw_path_values = list(CARTOGRAPHY_RAW_PATHS.values())
if len(set(raw_path_values)) != len(raw_path_values):
    raise ValueError(
        "Las rutas raw candidatas deben ser únicas. "
        f"Valor observado: {len(set(raw_path_values))} rutas únicas; esperado: {len(raw_path_values)}"
    )

candidate_path_values = list(CANDIDATE_INTERIM_PATHS.values())
if len(set(candidate_path_values)) != len(candidate_path_values):
    raise ValueError(
        "Las rutas interim candidatas deben ser únicas. "
        f"Valor observado: {len(set(candidate_path_values))} rutas únicas; esperado: {len(candidate_path_values)}"
    )

for dataset_id, legacy_path in LEGACY_INTERIM_PATHS.items():
    if not legacy_path.exists():
        raise _route_error(dataset_id, legacy_path, legacy_path.exists(), "el interim legacy debe existir")

for dataset_id, raw_path in CARTOGRAPHY_RAW_PATHS.items():
    if raw_path.suffix.lower() == ".zip":
        observed = route_contract.loc[
            route_contract["dataset_id"].eq(dataset_id), "n_shapefiles_en_zip"
        ].iloc[0]
        if observed != 1:
            raise _route_error(dataset_id, raw_path, observed, "el ZIP debe contener exactamente un .shp")

observed_ids = route_contract["dataset_id"].tolist()
if observed_ids != TARGET_DATASET_IDS:
    raise ValueError(
        "Los dataset_id de la tabla no coinciden con TARGET_DATASET_IDS. "
        f"Observado: {observed_ids}; esperado: {TARGET_DATASET_IDS}"
    )

route_contract

,dataset_id,archivo_raw_candidato,raw_existe,n_archivos_internos,n_shapefiles_en_zip,archivo_interim_candidato,interim_candidato_existe,archivo_interim_legacy,interim_legacy_existe
0,ser_geoportal_limite_ser,data/raw/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser.geojson,True,1,<NA>,data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_c...,False,data/interim/ser/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.par...,True
1,ser_geoportal_barrios_ser,data/raw/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser.geo...,True,1,<NA>,data/interim/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser...,False,data/interim/ser/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser_clean.p...,True
2,ser_geoportal_bandas_aparcamiento,data/raw/cartografia/ser_geoportal_bandas_aparcamiento/SHP_ZIP.zip,True,5,1,data/interim/cartografia/ser_geoportal_bandas_aparcamiento/ser_geoportal_ban...,False,data/interim/ser/ser_geoportal_bandas_aparcamiento/ser_geoportal_bandas_apar...,True
3,callejero_viales_vigentes,data/raw/cartografia/callejero_viales_vigentes/contexto_callejero_viales_vig...,True,5,1,data/interim/cartografia/callejero_viales_vigentes/callejero_viales_vigentes...,False,NaN,<NA>


Deben aparecer exactamente cuatro fuentes y todos los archivos raw deben estar disponibles. Las rutas previstas para las salidas limpias deben ser únicas y cada archivo ZIP debe contener un único shapefile inequívoco.

La columna `interim_candidato_existe` actúa como control de estado: inicialmente los outputs no existen, pero pasarán a estar disponibles conforme se complete la limpieza de cada fuente. Su existencia no debe impedir volver a ejecutar el notebook.

Si falta algún raw, una ruta está duplicada o un ZIP no contiene un shapefile inequívoco, el proceso debe detenerse antes de leer las geometrías. Si estas condiciones se cumplen, se autoriza la inspección estructural individual de las cuatro capas.

## 2. Funciones auxiliares geoespaciales

Se definen únicamente funciones generales necesarias para mostrar rutas relativas, normalizar nombres de columnas, leer GeoJSON, leer un shapefile contenido en ZIP, asegurar EPSG:25830 y resumir estructura y calidad geométrica.

Estas funciones no limpian ninguna fuente, no eliminan registros, no reparan geometrías y no deciden columnas finales. Las funciones específicas de límite, barrios, bandas y callejero se incorporarán en fases posteriores.

Las funciones específicas de limpieza se conservan dentro del bloque de cada fuente. Solo se incorporan aquí las utilidades compartidas estrictamente necesarias para reproducir la lógica ya validada.

In [3]:
def relpath(path: Path) -> str:
    try:
        return Path(path).relative_to(ROOT).as_posix()
    except ValueError:
        return str(path)


def strip_accents(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value)
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_key(value: Any) -> str:
    text = strip_accents(str(value).strip().lower())
    chars = []
    previous_was_separator = False
    for char in text:
        if char.isalnum():
            chars.append(char)
            previous_was_separator = False
        elif not previous_was_separator:
            chars.append("_")
            previous_was_separator = True
    return "".join(chars).strip("_")


def make_unique_columns(columns: list[Any]) -> list[str]:
    seen: dict[str, int] = {}
    unique_columns = []
    for column in columns:
        base = normalize_key(column)
        if not base:
            base = "column"
        count = seen.get(base, 0)
        unique = base if count == 0 else f"{base}_{count}"
        while unique in seen:
            count += 1
            unique = f"{base}_{count}"
        seen[base] = count + 1
        seen[unique] = 1
        unique_columns.append(unique)
    return unique_columns


def normalize_geo_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(f"Se esperaba un GeoDataFrame; observado: {type(gdf).__name__}")

    geometry_name = gdf.geometry.name
    if geometry_name not in gdf.columns:
        raise ValueError("El GeoDataFrame no tiene una columna geométrica activa presente en sus columnas.")

    non_geometry_columns = [column for column in gdf.columns if column != geometry_name]
    normalized_columns = make_unique_columns(["geometry", *non_geometry_columns])
    attribute_columns = normalized_columns[1:]

    attributes = pd.DataFrame(gdf.drop(columns=[geometry_name])).copy()
    attributes.columns = attribute_columns

    geometry = gpd.GeoSeries(
        gdf.geometry.copy(),
        index=gdf.index,
        crs=gdf.crs,
        name="geometry",
    )

    result = gpd.GeoDataFrame(attributes, geometry=geometry, crs=gdf.crs)
    result = result.set_geometry("geometry")
    return result


def read_geojson(path: Path, dataset_id: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            f"gpd.read_file no devolvió un GeoDataFrame: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            "no existe una geometría activa en el GeoDataFrame leído."
        )
    return gdf


def read_shp_zip(path: Path, dataset_id: str) -> gpd.GeoDataFrame:
    with zipfile.ZipFile(path) as archive:
        shp_members = [name for name in archive.namelist() if name.lower().endswith(".shp")]
        if len(shp_members) != 1:
            raise ValueError(
                f"Dataset: {dataset_id}; ruta: {relpath(path)}; valor observado: {len(shp_members)}; "
                "condición esperada: el ZIP debe contener exactamente un .shp"
            )
        shp_member = shp_members[0]
        with tempfile.TemporaryDirectory() as tmpdir:
            archive.extractall(tmpdir)
            shp_path = Path(tmpdir) / shp_member
            if not shp_path.exists():
                raise FileNotFoundError(
                    f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
                    f"no se localizó el shapefile extraído: {shp_member}"
                )
            gdf = gpd.read_file(shp_path)

    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            f"gpd.read_file no devolvió un GeoDataFrame: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            "no existe una geometría activa en el GeoDataFrame leído."
        )
    return gdf


def ensure_crs_25830(gdf: gpd.GeoDataFrame, dataset_id: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: CRS nulo; "
            "condición esperada: CRS declarado y transformable a EPSG:25830"
        )
    try:
        epsg = gdf.crs.to_epsg()
        if epsg == 25830:
            return gdf.copy()
        return gdf.to_crs(TARGET_CRS)
    except Exception as exc:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: {gdf.crs}; "
            "condición esperada: CRS transformable a EPSG:25830"
        ) from exc


def geometry_quality_summary(gdf: gpd.GeoDataFrame, dataset_id: str) -> dict[str, Any]:
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; se esperaba un GeoDataFrame; observado: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(f"Dataset: {dataset_id}; el GeoDataFrame no tiene geometría activa.")
    if gdf.crs is None:
        raise ValueError(f"Dataset: {dataset_id}; el CRS no está declarado.")

    geometry = gdf.geometry
    non_null_geometry = geometry[geometry.notna()]
    geometry_types = sorted(non_null_geometry.geom_type.dropna().unique().tolist())
    empty_count = int(non_null_geometry.is_empty.sum())
    invalid_count = int((~non_null_geometry.is_valid).sum())

    polygon_mask = non_null_geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    line_mask = non_null_geometry.geom_type.isin(["LineString", "MultiLineString"])
    area_total = non_null_geometry.loc[polygon_mask].area.sum() if polygon_mask.any() else pd.NA
    length_total = non_null_geometry.loc[line_mask].length.sum() if line_mask.any() else pd.NA

    return {
        "dataset_id": dataset_id,
        "n_filas": len(gdf),
        "n_columnas": len(gdf.columns),
        "columnas_normalizadas": list(gdf.columns),
        "crs": str(gdf.crs),
        "epsg": gdf.crs.to_epsg(),
        "tipos_geometria": geometry_types,
        "geometrias_nulas": int(geometry.isna().sum()),
        "geometrias_vacias": empty_count,
        "geometrias_invalidas": invalid_count,
        "area_total_m2_aprox": area_total,
        "longitud_total_m_aprox": length_total,
    }


def clean_text_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    )


def _normalise_decimal_text(value: Any) -> Any:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().replace("\xa0", "").replace(" ", "")
    if text == "" or text.lower() in {"nan", "none", "<na>"}:
        return pd.NA
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    return text


def to_numeric_series(s: pd.Series) -> pd.Series:
    text = clean_text_series(s).map(_normalise_decimal_text)
    return pd.to_numeric(text, errors="coerce")


def normalize_label(value: Any) -> str | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = strip_accents(str(value)).strip().lower()
    return " ".join(text.split())


SER_COLOR_ALIASES = {
    "043000255 azul": "azul",
    "077214010 verde": "verde",
    "081209246 alta rotacion": "alta rotacion",
    "255000000 rojo": "rojo",
    "255140000 naranja": "naranja",
    "azul": "azul",
    "verde": "verde",
    "alta rotacion": "alta rotacion",
    "rojo": "rojo",
    "naranja": "naranja",
    "gris": "gris",
}


def normalize_ser_color(value: Any) -> str | pd.NA:
    label = normalize_label(value)
    if pd.isna(label):
        return pd.NA
    return SER_COLOR_ALIASES.get(label, label)


SER_REGULATED_COLORS_NORM = {
    "azul",
    "verde",
    "alta rotacion",
    "rojo",
    "naranja",
}


def compose_barrio_code(cod_distrito: pd.Series, num_barrio: pd.Series) -> pd.Series:
    cod_distrito_num = pd.to_numeric(cod_distrito, errors="coerce")
    num_barrio_num = pd.to_numeric(num_barrio, errors="coerce")
    return (cod_distrito_num * 100 + num_barrio_num).round().astype("Int64")


def union_geometry(gdf: gpd.GeoDataFrame):
    return gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union


def ensure_parquet_engine() -> None:
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError("Para escribir Parquet instala pyarrow en el entorno activo.") from exc


CARTOGRAPHY_COLUMN_ALIASES = {
    "ser_geoportal_limite_ser": {},
    "ser_geoportal_barrios_ser": {
        "coddis": "cod_distrito",
        "nomdis": "distrito",
        "codbar": "num_barrio",
        "nombar": "barrio",
    },
    "ser_geoportal_bandas_aparcamiento": {
        "id": "id_banda",
        "bateria_li": "bateria_linea",
        "res_numpla": "numero_plazas",
        "texto_caje": "texto_cajetin",
    },
}


def prepare_cartography_source(
    gdf: gpd.GeoDataFrame,
    dataset_id: str,
) -> gpd.GeoDataFrame:
    if dataset_id not in CARTOGRAPHY_COLUMN_ALIASES:
        raise ValueError(
            f"Dataset: {dataset_id}; no existe configuración en CARTOGRAPHY_COLUMN_ALIASES."
        )
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; se esperaba un GeoDataFrame; observado: {type(gdf).__name__}"
        )
    geometry_name = gdf.geometry.name
    if geometry_name not in gdf.columns:
        raise ValueError(f"Dataset: {dataset_id}; el GeoDataFrame no tiene geometría activa.")

    aliases = CARTOGRAPHY_COLUMN_ALIASES[dataset_id]
    rename_map = {source: target for source, target in aliases.items() if source in gdf.columns}
    for source, target in rename_map.items():
        if source != target and target in gdf.columns:
            raise ValueError(
                f"Dataset: {dataset_id}; columnas observadas: {list(gdf.columns)}; "
                f"renombrar {source!r} a {target!r} produciría una colisión."
            )

    renamed_columns = [rename_map.get(column, column) for column in gdf.columns]
    duplicated = pd.Index(renamed_columns)[pd.Index(renamed_columns).duplicated()].tolist()
    if duplicated:
        raise ValueError(
            f"Dataset: {dataset_id}; columnas observadas: {list(gdf.columns)}; "
            f"la operación produciría columnas duplicadas: {duplicated}"
        )

    prepared = gdf.rename(columns=rename_map).copy()
    geometry_after = rename_map.get(geometry_name, geometry_name)
    prepared = prepared.set_geometry(geometry_after)
    if geometry_after != "geometry":
        prepared = prepared.rename_geometry("geometry")
    return gpd.GeoDataFrame(prepared, geometry="geometry", crs=gdf.crs)


def require_columns(
    df: pd.DataFrame,
    required: list[str],
    dataset_id: str,
) -> None:
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(
            f"Dataset: {dataset_id}; columnas observadas: {list(df.columns)}; "
            f"columnas requeridas: {required}; columnas ausentes: {missing}"
        )


## 3. Inspección estructural individual de los raw

Cada fuente se inspecciona antes de tomar decisiones de limpieza. No se presuponen esquemas: las columnas disponibles se observan después de leer cada raw y no se renombra ni deriva ninguna variable específica sin comprobar antes que existe.

La normalización realizada afecta únicamente a los nombres de columnas. No se muestran registros individuales ni geometrías. La inspección revisa estructura, CRS, tipos geométricos y problemas de calidad.

In [4]:
GEO_RAW: dict[str, gpd.GeoDataFrame] = {}
quality_rows = []

for dataset_id in TARGET_DATASET_IDS:
    raw_path = CARTOGRAPHY_RAW_PATHS[dataset_id]
    if not raw_path.exists():
        raise FileNotFoundError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: no existe; "
            "condición esperada: el raw candidato debe existir"
        )

    suffix = raw_path.suffix.lower()
    if suffix in {".geojson", ".json"}:
        raw_gdf = read_geojson(raw_path, dataset_id)
    elif suffix == ".zip":
        raw_gdf = read_shp_zip(raw_path, dataset_id)
    else:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: extensión {raw_path.suffix}; "
            "condición esperada: .geojson, .json o .zip"
        )

    observed_columns = list(raw_gdf.columns)
    if not observed_columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: 0 columnas; "
            "condición esperada: columnas estructurales observables"
        )

    normalized_gdf = normalize_geo_columns(raw_gdf)
    projected_gdf = ensure_crs_25830(normalized_gdf, dataset_id)

    if projected_gdf.empty:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: GeoDataFrame vacío; "
            "condición esperada: al menos una fila para inspección estructural"
        )

    GEO_RAW[dataset_id] = projected_gdf
    quality_rows.append(geometry_quality_summary(projected_gdf, dataset_id))

inspection_table = pd.DataFrame(quality_rows)

observed_geo_ids = list(GEO_RAW)
if observed_geo_ids != TARGET_DATASET_IDS:
    raise ValueError(
        "GEO_RAW no contiene exactamente los cuatro dataset_id esperados. "
        f"Observado: {observed_geo_ids}; esperado: {TARGET_DATASET_IDS}"
    )

if len(inspection_table) != 4:
    raise ValueError(
        "La tabla de inspección debe contener exactamente cuatro filas. "
        f"Valor observado: {len(inspection_table)}; esperado: 4"
    )

for dataset_id, gdf in GEO_RAW.items():
    observed_epsg = gdf.crs.to_epsg() if gdf.crs is not None else None
    if observed_epsg != 25830:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: EPSG {observed_epsg}; "
            "condición esperada: EPSG:25830"
        )

inspection_table

,dataset_id,n_filas,n_columnas,columnas_normalizadas,crs,epsg,tipos_geometria,geometrias_nulas,geometrias_vacias,geometrias_invalidas,area_total_m2_aprox,longitud_total_m_aprox
0,ser_geoportal_limite_ser,1,3,"[nombre, objectid, geometry]",EPSG:25830,25830,[Polygon],0,0,0,58686575.954764,<NA>
1,ser_geoportal_barrios_ser,67,6,"[coddis, nomdis, codbar, nombar, objectid, geometry]",EPSG:25830,25830,[Polygon],0,0,0,604455106.868571,<NA>
2,ser_geoportal_bandas_aparcamiento,87615,6,"[id, color, bateria_li, res_numpla, texto_caje, geometry]",EPSG:25830,25830,[LineString],0,0,0,<NA>,2185544.380116
3,callejero_viales_vigentes,9409,10,"[top_id, top_id_com, top_id_ter, tvia_id, top_dt_fch, top_dt_f_1, cv_tx_deno...",EPSG:25830,25830,"[MultiPolygon, Polygon]",122,0,37,89221620.640907,<NA>


### Lectura y criterio para continuar

La inspección confirma que las cuatro fuentes se leen correctamente y pueden expresarse en EPSG:25830.

- `ser_geoportal_limite_ser` contiene un único polígono, sin geometrías nulas, vacías o inválidas. Su estructura es coherente con su función como delimitación oficial del ámbito SER.
- `ser_geoportal_barrios_ser` contiene 67 polígonos válidos. El área bruta de la capa no debe interpretarse todavía como área SER efectiva, porque antes es necesario revisar sus atributos y distinguir los polígonos realmente pertenecientes al ámbito regulado.
- `ser_geoportal_bandas_aparcamiento` contiene 87.615 geometrías lineales, sin nulos, vacíos o geometrías inválidas. La capa es estructuralmente adecuada para iniciar la revisión de colores, plazas y geometrías de bandas.
- `callejero_viales_vigentes` contiene 9.409 geometrías poligonales o multipoligonales. Se detectan 122 geometrías nulas y 37 geometrías inválidas, por lo que su incorporación al mapa queda condicionada a diagnosticar estas incidencias y comprobar que pueden repararse o excluirse sin una pérdida espacial relevante.

La decisión global es **go condicionado**. Las tres capas Geoportal presentan una estructura suficiente para continuar con su limpieza individual. El callejero también puede mantenerse, pero requiere una fase específica de diagnóstico y reparación geométrica antes de construir el fondo vial.

A partir de este diagnóstico, las siguientes secciones preparan el límite SER, los barrios SER y las bandas de aparcamiento. La limpieza del callejero permanece fuera de esta fase y se abordará cuando las tres capas Geoportal hayan quedado reproducidas y validadas.

## 4. Limpieza de `ser_geoportal_limite_ser`

**Qué mide.** `ser_geoportal_limite_ser` contiene el polígono oficial del ámbito del Servicio de Estacionamiento Regulado.

**Uso en el TFM.** Sirve como geometría de referencia para validar si puntos o líneas SER caen dentro del área regulada y como base espacial para mapas posteriores.

**Columnas conservadas.** Se conservan `objectid`, `nombre` y `geometry`, porque identifican el polígono y su geometría oficial.

**Columnas descartadas.** No se añaden `dataset_id` ni `archivo_origen` al clean final, porque la fuente ya queda trazada por `data_catalog.csv` y por la ruta de salida.

**Validaciones.** Se comprueba número de geometrías, CRS, validez geométrica, geometrías nulas, área aproximada y nombres únicos. Esta fuente no requiere limpieza pesada: requiere validación y normalización geoespacial.


In [5]:
LIMITE_FINAL_COLUMNS = ["objectid", "nombre", "geometry"]

limite_input = prepare_cartography_source(
    GEO_RAW["ser_geoportal_limite_ser"],
    "ser_geoportal_limite_ser",
)
require_columns(
    limite_input,
    LIMITE_FINAL_COLUMNS,
    "ser_geoportal_limite_ser",
)


def clean_limite_ser(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    require_columns(gdf, LIMITE_FINAL_COLUMNS, "ser_geoportal_limite_ser")
    df = gdf.copy()
    out = gpd.GeoDataFrame({
        "objectid": pd.to_numeric(df["objectid"], errors="coerce").astype("Int64"),
        "nombre": clean_text_series(df["nombre"]),
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    return ensure_crs_25830(out, "ser_geoportal_limite_ser")


ser_geoportal_limite_ser_clean = clean_limite_ser(limite_input)
limite_geom_valid = ser_geoportal_limite_ser_clean.geometry.is_valid
limite_quality = pd.DataFrame([
    ("n_geometrias", int(len(ser_geoportal_limite_ser_clean)), "Número de geometrías del límite SER."),
    ("crs_epsg", int(ser_geoportal_limite_ser_clean.crs.to_epsg()), "Debe ser 25830 para medir distancias y áreas en metros."),
    ("n_geometrias_validas", int(limite_geom_valid.sum()), "Geometrías válidas."),
    ("n_geometrias_nulas", int(ser_geoportal_limite_ser_clean.geometry.isna().sum()), "Geometrías nulas."),
    ("area_aproximada_m2", float(ser_geoportal_limite_ser_clean.geometry.area.sum()), "Área total aproximada en m2."),
    ("nombres_unicos", ser_geoportal_limite_ser_clean["nombre"].dropna().unique().tolist(), "Nombres únicos de la capa."),
], columns=["check", "valor", "interpretacion"])

limite_quality

,check,valor,interpretacion
0,n_geometrias,1,Número de geometrías del límite SER.
1,crs_epsg,25830,Debe ser 25830 para medir distancias y áreas en metros.
2,n_geometrias_validas,1,Geometrías válidas.
3,n_geometrias_nulas,0,Geometrías nulas.
4,area_aproximada_m2,58686575.954764,Área total aproximada en m2.
5,nombres_unicos,[Zona S.E.R.],Nombres únicos de la capa.


**Lectura/decisión.** La capa contiene una única geometría válida, sin geometrías nulas, en EPSG:25830. El área aproximada es de 58.686.575,95 m² y el nombre único es `Zona S.E.R.`.

Con esta evidencia, el límite SER se acepta como geometría oficial de referencia. No se muestra una tabla limpia adicional porque el output final tiene una sola fila y su contenido ya queda suficientemente descrito por los checks.


## 5. Limpieza de `ser_geoportal_barrios_ser`

**Qué mide.** `ser_geoportal_barrios_ser` contiene los polígonos de barrios dentro del ámbito SER.

**Uso en el TFM.** Permite dividir el mapa SER por barrios y habilita futuras agregaciones espaciales por barrio. No sustituye a `ser_calles_plazas` como fuente de capacidad.

**Columnas conservadas.** Se conservan `cod_distrito`, `distrito`, `num_barrio`, `cod_barrio`, `barrio`, `objectid` y `geometry`. La regla `cod_barrio = cod_distrito * 100 + num_barrio` se usa como armonización del código compuesto de barrio.

**Columnas descartadas.** Se excluye del clean el polígono “No está en la zona SER”, porque no representa un barrio SER utilizable. También se descartan los flags de diagnóstico y la trazabilidad redundante, ya que solo se utilizan para justificar la selección de los polígonos que forman el output limpio.

**Validaciones.** Se comprueban conteos de polígonos, áreas, coherencia entre la unión de barrios y el límite SER, y duplicados de `cod_barrio`. Si aparece un duplicado, se diagnostica geométricamente antes de decidir si se elimina, se conserva o se pospone su disolución.


In [6]:
BARRIOS_INPUT_COLUMNS = [
    "cod_distrito",
    "distrito",
    "num_barrio",
    "barrio",
    "objectid",
    "geometry",
]

BARRIOS_FINAL_COLUMNS = [
    "cod_distrito",
    "distrito",
    "num_barrio",
    "cod_barrio",
    "barrio",
    "objectid",
    "geometry",
]

TOL_AREA_M2 = 1.0

barrios_input = prepare_cartography_source(
    GEO_RAW["ser_geoportal_barrios_ser"],
    "ser_geoportal_barrios_ser",
)
require_columns(
    barrios_input,
    BARRIOS_INPUT_COLUMNS,
    "ser_geoportal_barrios_ser",
)


def clean_barrios_ser_diagnostic(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    require_columns(gdf, BARRIOS_INPUT_COLUMNS, "ser_geoportal_barrios_ser")
    df = gdf.copy()
    cod_distrito = pd.to_numeric(df["cod_distrito"], errors="coerce").astype("Int64")
    num_barrio = pd.to_numeric(df["num_barrio"], errors="coerce").astype("Int64")
    cod_barrio = compose_barrio_code(cod_distrito, num_barrio)
    barrio = clean_text_series(df["barrio"])
    barrio_norm = barrio.map(normalize_label)
    flag_en_zona_ser = barrio_norm.ne("no esta en la zona ser") & cod_barrio.notna()
    diagnostic = gpd.GeoDataFrame({
        "cod_distrito": cod_distrito,
        "distrito": clean_text_series(df["distrito"]),
        "num_barrio": num_barrio,
        "cod_barrio": cod_barrio,
        "barrio": barrio,
        "objectid": pd.to_numeric(df["objectid"], errors="coerce").astype("Int64"),
        "flag_en_zona_ser": flag_en_zona_ser.fillna(False),
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    return ensure_crs_25830(diagnostic, "ser_geoportal_barrios_ser")


def fmt_int(value: int) -> int:
    return int(value)


def fmt_m2(value: float) -> str:
    return f"{float(value):.3f}"


def fmt_pct(value: float) -> str:
    return f"{float(value):.9f}"


ser_geoportal_barrios_ser_diagnostic = clean_barrios_ser_diagnostic(barrios_input)
barrios_en_zona = ser_geoportal_barrios_ser_diagnostic["flag_en_zona_ser"]
barrios_candidate_clean = ser_geoportal_barrios_ser_diagnostic.loc[barrios_en_zona, BARRIOS_FINAL_COLUMNS].copy()

barrios_union_geom = union_geometry(barrios_candidate_clean)
limite_geom = union_geometry(ser_geoportal_limite_ser_clean)
area_union_barrios_ser_m2 = float(barrios_union_geom.area)
area_limite_ser_m2 = float(limite_geom.area)
diferencia_area_m2 = area_union_barrios_ser_m2 - area_limite_ser_m2
diferencia_area_pct = float(diferencia_area_m2 / area_limite_ser_m2 * 100) if area_limite_ser_m2 else float("nan")

barrios_quality = pd.DataFrame([
    ("n_poligonos_total", fmt_int(len(ser_geoportal_barrios_ser_diagnostic)), "Polígonos recibidos en raw."),
    ("n_poligonos_en_zona_ser", fmt_int(barrios_en_zona.sum()), "Polígonos SER reales que pasan a candidato clean."),
    ("n_poligonos_no_ser", fmt_int((~barrios_en_zona).sum()), "Polígonos no SER excluidos del clean."),
    ("area_total_m2", fmt_m2(ser_geoportal_barrios_ser_diagnostic.geometry.area.sum()), "Área total del raw."),
    ("area_en_zona_ser_m2", fmt_m2(ser_geoportal_barrios_ser_diagnostic.loc[barrios_en_zona].geometry.area.sum()), "Área sumada de polígonos SER conservados."),
    ("area_union_barrios_ser_m2", fmt_m2(area_union_barrios_ser_m2), "Área de la unión geométrica de barrios SER limpios."),
    ("area_limite_ser_m2", fmt_m2(area_limite_ser_m2), "Área del límite SER oficial."),
    ("diferencia_area_m2", fmt_m2(diferencia_area_m2), "Diferencia unión barrios SER menos límite SER."),
    ("diferencia_area_pct", fmt_pct(diferencia_area_pct), "Diferencia relativa sobre área del límite SER."),
    ("duplicados_cod_barrio_en_zona_ser", fmt_int(barrios_candidate_clean.duplicated("cod_barrio").sum()), "Duplicados de código compuesto dentro de zona SER."),
], columns=["check", "valor", "interpretacion"])

barrios_quality

,check,valor,interpretacion
0,n_poligonos_total,67,Polígonos recibidos en raw.
1,n_poligonos_en_zona_ser,66,Polígonos SER reales que pasan a candidato clean.
2,n_poligonos_no_ser,1,Polígonos no SER excluidos del clean.
3,area_total_m2,604455106.869,Área total del raw.
4,area_en_zona_ser_m2,58686575.971,Área sumada de polígonos SER conservados.
5,area_union_barrios_ser_m2,58686575.949,Área de la unión geométrica de barrios SER limpios.
6,area_limite_ser_m2,58686575.955,Área del límite SER oficial.
7,diferencia_area_m2,-0.005,Diferencia unión barrios SER menos límite SER.
8,diferencia_area_pct,-0.000000009,Diferencia relativa sobre área del límite SER.
9,duplicados_cod_barrio_en_zona_ser,1,Duplicados de código compuesto dentro de zona SER.


**Lectura/decisión.** El raw contiene 67 polígonos, de los cuales 66 corresponden a barrios SER reales y 1 corresponde a “No está en la zona SER”. Ese polígono no-SER se excluye del clean porque inflaría el área total y no debe intervenir en mapas ni agregaciones SER.

La unión geométrica de los barrios SER limpios tiene un área de 58.686.575,949 m², prácticamente idéntica al límite SER oficial, con una diferencia de -0,005 m² (-0,000000009 %). Esto valida que, tras excluir el polígono no-SER, la cobertura espacial de barrios SER cuadra con el límite oficial.

Se detecta un duplicado de `cod_barrio` dentro de zona SER, por lo que se realiza un diagnóstico específico antes de cerrar el clean.


### 5.1. Diagnóstico de duplicados de `cod_barrio`

El duplicado detectado corresponde al código `904`, asociado a `Valdezarza` y `Valdezarza Fase III`. Esta revisión comprueba si el duplicado implica un error geométrico real o si se trata de dos piezas territoriales con el mismo código compuesto.

Se comprueba: si las geometrías se tocan o intersectan, si existe solape relevante, si una contiene a la otra, si son disjuntas, y si la unión queda dentro del límite SER.


In [7]:
duplicados_barrios_ser = (
    barrios_candidate_clean
    .assign(area_m2=lambda df: df.geometry.area)
    .loc[lambda df: df.duplicated("cod_barrio", keep=False)]
    .sort_values(["cod_barrio", "barrio"])
)

if not duplicados_barrios_ser.empty:
    display(duplicados_barrios_ser.drop(columns="geometry"))

barrios_904 = barrios_candidate_clean.loc[barrios_candidate_clean["cod_barrio"].eq(904)].copy()
if len(barrios_904) == 2:
    geom_a, geom_b = barrios_904.geometry.iloc[0], barrios_904.geometry.iloc[1]
    union_904 = geom_a.union(geom_b)
    area_interseccion_m2 = float(geom_a.intersection(geom_b).area)
    cod_904_geometria = pd.DataFrame([{
        "cod_barrio": 904,
        "barrio_a": barrios_904["barrio"].iloc[0],
        "barrio_b": barrios_904["barrio"].iloc[1],
        "area_a_m2": float(geom_a.area),
        "area_b_m2": float(geom_b.area),
        "se_tocan_o_intersectan": bool(geom_a.intersects(geom_b)),
        "area_interseccion_m2": area_interseccion_m2,
        "solape_relevante": bool(area_interseccion_m2 > TOL_AREA_M2),
        "a_contiene_b": bool(geom_a.contains(geom_b)),
        "b_contiene_a": bool(geom_b.contains(geom_a)),
        "son_disjuntas": bool(geom_a.disjoint(geom_b)),
        "area_union_m2": float(union_904.area),
        "area_union_dentro_limite_ser_m2": float(union_904.intersection(limite_geom).area),
        "diferencia_union_vs_limite_intersec_m2": float(union_904.area - union_904.intersection(limite_geom).area),
    }])
elif len(barrios_904) > 0:
    cod_904_geometria = pd.DataFrame([{
        "cod_barrio": 904,
        "n_poligonos": int(len(barrios_904)),
        "nota": "No hay exactamente dos polígonos con cod_barrio 904; revisar manualmente si cambia el raw.",
    }])
else:
    cod_904_geometria = pd.DataFrame()

if not cod_904_geometria.empty:
    display(cod_904_geometria)


,cod_distrito,distrito,num_barrio,cod_barrio,barrio,objectid,area_m2
46,9,Moncloa - Aravaca,4,904,Valdezarza,47,359529.629332
65,9,Moncloa - Aravaca,4,904,Valdezarza Fase III,67,209555.978181


,cod_barrio,barrio_a,barrio_b,area_a_m2,area_b_m2,se_tocan_o_intersectan,area_interseccion_m2,solape_relevante,a_contiene_b,b_contiene_a,son_disjuntas,area_union_m2,area_union_dentro_limite_ser_m2,diferencia_union_vs_limite_intersec_m2
0,904,Valdezarza,Valdezarza Fase III,359529.629332,209555.978181,True,0.002087,False,False,False,False,569085.605426,569085.604794,0.000631


**Lectura/decisión.** El duplicado `904` tiene dos geometrías con áreas de 359.529,63 m² y 209.555,98 m². Aunque `se_tocan_o_intersectan = True`, el área de intersección es de solo 0,002087 m², por debajo de la tolerancia de 1 m²; por tanto, `solape_relevante = False`.

Ninguna geometría contiene a la otra y la diferencia entre el área de la unión y su intersección con el límite SER es residual. La decisión es conservar ambas geometrías en el clean, sin disolver ni eliminar. Si más adelante se necesita un único polígono por `cod_barrio`, la disolución se hará en un notebook posterior y deberá documentarse explícitamente.


In [8]:
ser_geoportal_barrios_ser_clean = barrios_candidate_clean.copy()

if list(ser_geoportal_barrios_ser_clean.columns) != BARRIOS_FINAL_COLUMNS:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; columnas finales inesperadas. "
        f"Observado: {list(ser_geoportal_barrios_ser_clean.columns)}; esperado: {BARRIOS_FINAL_COLUMNS}"
    )
if len(ser_geoportal_barrios_ser_clean) != 66:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; número de filas inesperado. "
        f"Observado: {len(ser_geoportal_barrios_ser_clean)}; esperado: 66"
    )
observed_epsg = ser_geoportal_barrios_ser_clean.crs.to_epsg() if ser_geoportal_barrios_ser_clean.crs is not None else None
if observed_epsg != 25830:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; EPSG inesperado. "
        f"Observado: {observed_epsg}; esperado: 25830"
    )
if ser_geoportal_barrios_ser_clean.geometry.isna().sum() != 0:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; geometrías nulas inesperadas. "
        f"Observado: {int(ser_geoportal_barrios_ser_clean.geometry.isna().sum())}; esperado: 0"
    )
invalid_count = int((~ser_geoportal_barrios_ser_clean.geometry.is_valid & ser_geoportal_barrios_ser_clean.geometry.notna()).sum())
if invalid_count != 0:
    raise ValueError(
        "Dataset: ser_geoportal_barrios_ser; geometrías inválidas inesperadas. "
        f"Observado: {invalid_count}; esperado: 0"
    )

## 6. Limpieza de `ser_geoportal_bandas_aparcamiento`

**Qué mide.** `ser_geoportal_bandas_aparcamiento` contiene la geometría lineal de las bandas de aparcamiento SER. Es la fuente que permite representar en un mapa las líneas coloreadas de plazas reguladas.

**Uso en el TFM.** Se mantiene como capa cartográfica de oferta física regulada y como contraste espacial frente a `ser_calles_plazas`. No es el denominador principal del modelo: la capacidad tabular histórica sigue viniendo de `ser_calles_plazas`.

**Columnas conservadas.** Se conservan `id_banda`, `color`, `numero_plazas` y `geometry`, porque identifican la banda, el tipo de plaza, el número de plazas y la línea que se dibujará.

**Columnas descartadas.** Se descartan `texto_cajetin`, `bateria_linea`, `longitud_m`, flags y trazabilidad de origen. `longitud_m` no se guarda porque puede recalcularse desde la geometría si se necesita en un análisis posterior.

**Validaciones.** Se comprueban colores regulados, bandas grises, plazas nulas/cero/negativas, geometrías inválidas y posición respecto al límite SER. La posición espacial se evalúa con criterio estricto y con un buffer de 5 m: el buffer evita sobrerreaccionar ante líneas situadas en el borde del polígono o desplazadas levemente por precisión cartográfica.

En esta limpieza interim no se eliminan bandas reguladas solo por quedar fuera del límite con buffer. Si tienen color SER válido, plazas informadas y geometría válida, se conservan y la incidencia espacial queda diagnosticada para el notebook posterior de mapas.


In [9]:
BANDAS_FINAL_COLUMNS = ["id_banda", "color", "numero_plazas", "geometry"]

bandas_input = prepare_cartography_source(
    GEO_RAW["ser_geoportal_bandas_aparcamiento"],
    "ser_geoportal_bandas_aparcamiento",
)
require_columns(
    bandas_input,
    [
        "id_banda",
        "color",
        "bateria_linea",
        "numero_plazas",
        "texto_cajetin",
        "geometry",
    ],
    "ser_geoportal_bandas_aparcamiento",
)


def diagnose_bandas_aparcamiento(gdf: gpd.GeoDataFrame, limite_geom) -> gpd.GeoDataFrame:
    require_columns(
        gdf,
        [
            "id_banda",
            "color",
            "bateria_linea",
            "numero_plazas",
            "texto_cajetin",
            "geometry",
        ],
        "ser_geoportal_bandas_aparcamiento",
    )
    df = gdf.copy()
    color_norm = clean_text_series(df["color"]).map(normalize_ser_color)
    numero_plazas = to_numeric_series(df["numero_plazas"]).round().astype("Int64")
    limite_geom_buffer_5m_local = limite_geom.buffer(5)
    diagnostic = gpd.GeoDataFrame({
        "id_banda": pd.to_numeric(df["id_banda"], errors="coerce").astype("Int64"),
        "color": color_norm.map(lambda x: str(x).replace(" ", "_") if pd.notna(x) else pd.NA).astype("string"),
        "color_norm_diagnostico": color_norm,
        "bateria_linea": clean_text_series(df["bateria_linea"]),
        "numero_plazas": numero_plazas,
        "texto_cajetin": clean_text_series(df["texto_cajetin"]),
        "longitud_m": df.geometry.length,
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    diagnostic = ensure_crs_25830(diagnostic, "ser_geoportal_bandas_aparcamiento")
    diagnostic["flag_color_gris"] = diagnostic["color_norm_diagnostico"].eq("gris").fillna(False)
    diagnostic["flag_color_ser_regulado"] = diagnostic["color_norm_diagnostico"].isin(SER_REGULATED_COLORS_NORM).fillna(False)
    diagnostic["flag_plazas_nulas"] = diagnostic["numero_plazas"].isna()
    diagnostic["flag_plazas_cero"] = diagnostic["numero_plazas"].fillna(-1).eq(0)
    diagnostic["flag_plazas_negativas"] = diagnostic["numero_plazas"].fillna(0).lt(0)
    diagnostic["flag_geom_invalida"] = ~diagnostic.geometry.is_valid | diagnostic.geometry.isna()
    diagnostic["flag_fuera_limite_estricto"] = ~diagnostic.geometry.intersects(limite_geom)
    diagnostic["flag_fuera_limite_buffer_5m"] = ~diagnostic.geometry.intersects(limite_geom_buffer_5m_local)
    return diagnostic


limite_geom = union_geometry(ser_geoportal_limite_ser_clean)
ser_geoportal_bandas_aparcamiento_diagnostic = diagnose_bandas_aparcamiento(
    bandas_input, limite_geom
)
bandas_diag = ser_geoportal_bandas_aparcamiento_diagnostic
bandas_keep = (
    bandas_diag["flag_color_ser_regulado"]
    & bandas_diag["numero_plazas"].notna()
    & ~bandas_diag["flag_geom_invalida"]
)
bandas_quality = pd.DataFrame([
    ("n_bandas_raw", int(len(bandas_diag)), "Bandas recibidas."),
    ("n_bandas_candidato_clean", int(bandas_keep.sum()), "Bandas SER reguladas válidas por color, plazas y geometría; el límite queda como diagnóstico."),
    ("n_bandas_sin_color", int(bandas_diag["color_norm_diagnostico"].isna().sum()), "Bandas sin color normalizable; se excluyen del clean."),
    ("n_bandas_gris", int(bandas_diag["flag_color_gris"].sum()), "Bandas grises diagnosticadas; se excluyen por no ser color SER regulado objetivo."),
    ("plazas_gris", int(bandas_diag.loc[bandas_diag["flag_color_gris"], "numero_plazas"].fillna(0).sum()), "Plazas asociadas a bandas grises."),
    ("n_plazas_nulas", int(bandas_diag["flag_plazas_nulas"].sum()), "Registros con numero_plazas nulo; se excluyen del clean."),
    ("n_plazas_cero", int(bandas_diag["flag_plazas_cero"].sum()), "Registros con cero plazas."),
    ("n_plazas_negativas", int(bandas_diag["flag_plazas_negativas"].sum()), "Registros con plazas negativas."),
    ("n_geometrias_invalidas", int(bandas_diag["flag_geom_invalida"].sum()), "Geometrías nulas o inválidas."),
    ("n_bandas_gris_fuera_limite_estricto", int((bandas_diag["flag_color_gris"] & bandas_diag["flag_fuera_limite_estricto"]).sum()), "Bandas grises fuera del límite estricto."),
    ("n_bandas_gris_fuera_limite_buffer_5m", int((bandas_diag["flag_color_gris"] & bandas_diag["flag_fuera_limite_buffer_5m"]).sum()), "Bandas grises fuera del límite con buffer 5 m."),
    ("n_bandas_reguladas_fuera_limite_estricto", int((bandas_diag["flag_color_ser_regulado"] & bandas_diag["flag_fuera_limite_estricto"]).sum()), "Bandas reguladas fuera del límite estricto."),
    ("n_bandas_reguladas_fuera_limite_buffer_5m", int((bandas_diag["flag_color_ser_regulado"] & bandas_diag["flag_fuera_limite_buffer_5m"]).sum()), "Bandas reguladas fuera incluso con buffer 5 m; se conservan como diagnóstico para el mapa."),
], columns=["check", "valor", "interpretacion"])

bandas_quality

,check,valor,interpretacion
0,n_bandas_raw,87615,Bandas recibidas.
1,n_bandas_candidato_clean,34450,"Bandas SER reguladas válidas por color, plazas y geometría; el límite queda ..."
2,n_bandas_sin_color,2,Bandas sin color normalizable; se excluyen del clean.
3,n_bandas_gris,53163,Bandas grises diagnosticadas; se excluyen por no ser color SER regulado obje...
4,plazas_gris,331380,Plazas asociadas a bandas grises.
5,n_plazas_nulas,2,Registros con numero_plazas nulo; se excluyen del clean.
6,n_plazas_cero,0,Registros con cero plazas.
7,n_plazas_negativas,0,Registros con plazas negativas.
8,n_geometrias_invalidas,0,Geometrías nulas o inválidas.
9,n_bandas_gris_fuera_limite_estricto,53151,Bandas grises fuera del límite estricto.


**Lectura/decisión.** El clean conserva las bandas con color SER regulado, `numero_plazas` informado y geometría válida. Se excluyen las bandas sin color normalizable y los registros con `numero_plazas` nulo, porque no pueden simbolizarse ni aportar capacidad fiable en la capa cartográfica.

Las bandas grises se excluyen porque no pertenecen a los colores SER regulados objetivo del TFM (`azul`, `verde`, `alta_rotacion`, `rojo`, `naranja`). No se eliminan porque todas estén fuera del límite: una parte queda dentro o cerca del ámbito SER. La justificación correcta es semántica/cartográfica, no puramente espacial.

En cuanto al límite SER, el notebook distingue entre fuera del límite estricto y fuera con buffer de 5 m. Las bandas reguladas que siguen fuera incluso con buffer se mantienen en el clean interim porque tienen color SER válido y geometría válida. Su tratamiento cartográfico se decidirá en la fase de construcción del mapa, donde podrán excluirse únicamente de una representación concreta si la inspección visual demuestra que son incoherentes. Esta decisión no modificará el output limpio.


In [10]:
ser_geoportal_bandas_aparcamiento_clean = bandas_diag.loc[
    bandas_keep,
    BANDAS_FINAL_COLUMNS,
].copy()

if list(ser_geoportal_bandas_aparcamiento_clean.columns) != BANDAS_FINAL_COLUMNS:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; columnas finales inesperadas. "
        f"Observado: {list(ser_geoportal_bandas_aparcamiento_clean.columns)}; esperado: {BANDAS_FINAL_COLUMNS}"
    )
if len(ser_geoportal_bandas_aparcamiento_clean) != 34450:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; número de filas inesperado. "
        f"Observado: {len(ser_geoportal_bandas_aparcamiento_clean)}; esperado: 34450"
    )
observed_epsg = ser_geoportal_bandas_aparcamiento_clean.crs.to_epsg() if ser_geoportal_bandas_aparcamiento_clean.crs is not None else None
if observed_epsg != 25830:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; EPSG inesperado. "
        f"Observado: {observed_epsg}; esperado: 25830"
    )
if ser_geoportal_bandas_aparcamiento_clean.geometry.isna().sum() != 0:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; geometrías nulas inesperadas. "
        f"Observado: {int(ser_geoportal_bandas_aparcamiento_clean.geometry.isna().sum())}; esperado: 0"
    )
invalid_count = int((~ser_geoportal_bandas_aparcamiento_clean.geometry.is_valid & ser_geoportal_bandas_aparcamiento_clean.geometry.notna()).sum())
if invalid_count != 0:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; geometrías inválidas inesperadas. "
        f"Observado: {invalid_count}; esperado: 0"
    )
if ser_geoportal_bandas_aparcamiento_clean["numero_plazas"].isna().sum() != 0:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; numero_plazas nulos inesperados. "
        f"Observado: {int(ser_geoportal_bandas_aparcamiento_clean['numero_plazas'].isna().sum())}; esperado: 0"
    )
allowed_clean_colors = {"azul", "verde", "alta_rotacion", "rojo", "naranja"}
observed_colors = set(ser_geoportal_bandas_aparcamiento_clean["color"].dropna().astype(str).unique())
unexpected_colors = sorted(observed_colors - allowed_clean_colors)
if unexpected_colors:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; colores finales fuera de catálogo. "
        f"Observado: {unexpected_colors}; esperado: {sorted(allowed_clean_colors)}"
    )

## 7. Control de equivalencia y escritura

Las tres capas limpias se comparan con las salidas de referencia obtenidas mediante el procesamiento previamente validado. El objetivo no es redefinir la limpieza, sino comprobar que las nuevas rutas y dependencias reproducen exactamente sus resultados.

El control revisa esquema, dimensiones, CRS, atributos, geometrías y magnitudes espaciales agregadas. Solo si las tres capas superan todas las comprobaciones se escriben los nuevos Parquet en `data/interim/cartografia/`.

La escritura se limita a las tres fuentes Geoportal tratadas en esta fase. No se generan mapas, agregados, joins ni archivos `processed`.

In [11]:
geoportal_clean_outputs = {
    "ser_geoportal_limite_ser": ser_geoportal_limite_ser_clean,
    "ser_geoportal_barrios_ser": ser_geoportal_barrios_ser_clean,
    "ser_geoportal_bandas_aparcamiento": ser_geoportal_bandas_aparcamiento_clean,
}

EXPECTED_GEOPORTAL_SHAPES = {
    "ser_geoportal_limite_ser": (1, 3),
    "ser_geoportal_barrios_ser": (66, 7),
    "ser_geoportal_bandas_aparcamiento": (34450, 4),
}


def _check_row(
    dataset_id: str,
    check: str,
    ok: bool,
    valor_actual: Any,
    valor_referencia: Any,
    criterio: str,
) -> dict[str, Any]:
    return {
        "dataset_id": dataset_id,
        "check": check,
        "resultado": "OK" if ok else "ERROR",
        "valor_actual": valor_actual,
        "valor_referencia": valor_referencia,
        "criterio": criterio,
    }


def _topological_equal(current: gpd.GeoSeries, reference: gpd.GeoSeries) -> bool:
    if len(current) != len(reference):
        return False
    return all(geom_current.equals(geom_reference) for geom_current, geom_reference in zip(current, reference))


def compare_geodataframes(
    current: gpd.GeoDataFrame,
    reference: gpd.GeoDataFrame,
    dataset_id: str,
) -> list[dict[str, Any]]:
    rows = []
    current_is_geo = isinstance(current, gpd.GeoDataFrame)
    reference_is_geo = isinstance(reference, gpd.GeoDataFrame)
    rows.append(_check_row(
        dataset_id,
        "tipo_geodataframe",
        current_is_geo and reference_is_geo,
        type(current).__name__,
        type(reference).__name__,
        "ambos objetos deben ser GeoDataFrame",
    ))
    if not current_is_geo or not reference_is_geo:
        return rows

    current_columns = list(current.columns)
    reference_columns = list(reference.columns)
    rows.append(_check_row(
        dataset_id,
        "columnas_orden",
        current_columns == reference_columns,
        current_columns,
        reference_columns,
        "mismas columnas y mismo orden",
    ))

    current_shape = current.shape
    reference_shape = reference.shape
    expected_shape = EXPECTED_GEOPORTAL_SHAPES[dataset_id]
    rows.append(_check_row(
        dataset_id,
        "dimension_referencia",
        current_shape == reference_shape,
        current_shape,
        reference_shape,
        "misma dimensión que la referencia",
    ))
    rows.append(_check_row(
        dataset_id,
        "dimension_esperada_validada",
        current_shape == expected_shape,
        current_shape,
        expected_shape,
        "dimensión igual al resultado previamente validado",
    ))

    current_epsg = current.crs.to_epsg() if current.crs is not None else None
    reference_epsg = reference.crs.to_epsg() if reference.crs is not None else None
    rows.append(_check_row(
        dataset_id,
        "epsg",
        current_epsg == reference_epsg,
        current_epsg,
        reference_epsg,
        "mismo EPSG",
    ))

    non_geometry_columns = [column for column in current_columns if column != current.geometry.name]
    try:
        pd.testing.assert_frame_equal(
            current[non_geometry_columns].reset_index(drop=True),
            reference[non_geometry_columns].reset_index(drop=True),
            check_dtype=False,
        )
        attributes_ok = True
        attributes_error = "OK"
    except AssertionError as exc:
        attributes_ok = False
        attributes_error = str(exc).splitlines()[0]
    rows.append(_check_row(
        dataset_id,
        "atributos_no_geometricos",
        attributes_ok,
        attributes_error,
        "OK",
        "atributos iguales con índices reiniciados y check_dtype=False",
    ))

    geometry_ok = _topological_equal(
        current.geometry.reset_index(drop=True),
        reference.geometry.reset_index(drop=True),
    )
    rows.append(_check_row(
        dataset_id,
        "geometrias_topologicas_fila_a_fila",
        geometry_ok,
        geometry_ok,
        True,
        "geometry.equals() fila a fila en el orden original",
    ))

    if dataset_id in {"ser_geoportal_limite_ser", "ser_geoportal_barrios_ser"}:
        current_area = float(current.geometry.area.sum())
        reference_area = float(reference.geometry.area.sum())
        rows.append(_check_row(
            dataset_id,
            "area_total_m2",
            abs(current_area - reference_area) <= 1e-6,
            current_area,
            reference_area,
            "diferencia absoluta <= 1e-6 m2",
        ))
    elif dataset_id == "ser_geoportal_bandas_aparcamiento":
        current_length = float(current.geometry.length.sum())
        reference_length = float(reference.geometry.length.sum())
        rows.append(_check_row(
            dataset_id,
            "longitud_total_m",
            abs(current_length - reference_length) <= 1e-6,
            current_length,
            reference_length,
            "diferencia absoluta <= 1e-6 m",
        ))

    return rows


parity_rows = []
legacy_references: dict[str, gpd.GeoDataFrame] = {}
for dataset_id, current in geoportal_clean_outputs.items():
    legacy_path = LEGACY_INTERIM_PATHS[dataset_id]
    if not legacy_path.exists():
        raise FileNotFoundError(
            f"Dataset: {dataset_id}; ruta: {relpath(legacy_path)}; valor observado: no existe; "
            "condición esperada: Parquet legacy disponible para equivalencia"
        )
    reference = gpd.read_parquet(legacy_path)
    legacy_references[dataset_id] = reference
    parity_rows.extend(compare_geodataframes(current, reference, dataset_id))

parity_check = pd.DataFrame(parity_rows)
display(parity_check)

failed_parity = parity_check.loc[parity_check["resultado"].ne("OK"), ["dataset_id", "check"]]
if not failed_parity.empty:
    raise ValueError(
        "Fallo en el control de equivalencia; no se escribe ningún output. "
        f"Checks fallidos: {failed_parity.to_dict(orient='records')}"
    )

ensure_parquet_engine()
write_rows = []
for dataset_id, current in geoportal_clean_outputs.items():
    output_path = CANDIDATE_INTERIM_PATHS[dataset_id]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    current.to_parquet(output_path, index=False)

    roundtrip = gpd.read_parquet(output_path)
    reference_checks = compare_geodataframes(current, legacy_references[dataset_id], dataset_id)
    roundtrip_checks = compare_geodataframes(roundtrip, current, dataset_id)
    reference_ok = all(row["resultado"] == "OK" for row in reference_checks)
    roundtrip_ok = all(row["resultado"] == "OK" for row in roundtrip_checks)
    if not reference_ok or not roundtrip_ok:
        failed_after_write = [
            {"dataset_id": row["dataset_id"], "check": row["check"]}
            for row in [*reference_checks, *roundtrip_checks]
            if row["resultado"] != "OK"
        ]
        raise ValueError(
            "Fallo en el control posterior a escritura. "
            f"Checks fallidos: {failed_after_write}"
        )

    write_rows.append({
        "dataset_id": dataset_id,
        "archivo_interim": relpath(output_path),
        "shape_escrita": roundtrip.shape,
        "epsg": roundtrip.crs.to_epsg() if roundtrip.crs is not None else None,
        "size_mb": round(output_path.stat().st_size / 1024 / 1024, 3),
        "equivalencia_referencia": "OK" if reference_ok else "ERROR",
        "roundtrip": "OK" if roundtrip_ok else "ERROR",
        "estado_escritura": "OK",
    })

write_check = pd.DataFrame(write_rows)
display(write_check)

,dataset_id,check,resultado,valor_actual,valor_referencia,criterio
0,ser_geoportal_limite_ser,tipo_geodataframe,OK,GeoDataFrame,GeoDataFrame,ambos objetos deben ser GeoDataFrame
1,ser_geoportal_limite_ser,columnas_orden,OK,"[objectid, nombre, geometry]","[objectid, nombre, geometry]",mismas columnas y mismo orden
2,ser_geoportal_limite_ser,dimension_referencia,OK,"(1, 3)","(1, 3)",misma dimensión que la referencia
3,ser_geoportal_limite_ser,dimension_esperada_validada,OK,"(1, 3)","(1, 3)",dimensión igual al resultado previamente validado
4,ser_geoportal_limite_ser,epsg,OK,25830,25830,mismo EPSG
5,ser_geoportal_limite_ser,atributos_no_geometricos,OK,OK,OK,atributos iguales con índices reiniciados y check_dtype=False
6,ser_geoportal_limite_ser,geometrias_topologicas_fila_a_fila,OK,True,True,geometry.equals() fila a fila en el orden original
7,ser_geoportal_limite_ser,area_total_m2,OK,58686575.954764,58686575.954764,diferencia absoluta <= 1e-6 m2
8,ser_geoportal_barrios_ser,tipo_geodataframe,OK,GeoDataFrame,GeoDataFrame,ambos objetos deben ser GeoDataFrame
9,ser_geoportal_barrios_ser,columnas_orden,OK,"[cod_distrito, distrito, num_barrio, cod_barrio, barrio, objectid, geometry]","[cod_distrito, distrito, num_barrio, cod_barrio, barrio, objectid, geometry]",mismas columnas y mismo orden


,dataset_id,archivo_interim,shape_escrita,epsg,size_mb,equivalencia_referencia,roundtrip,estado_escritura
0,ser_geoportal_limite_ser,data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_c...,"(1, 3)",25830,0.043,OK,OK,OK
1,ser_geoportal_barrios_ser,data/interim/cartografia/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser...,"(66, 7)",25830,0.081,OK,OK,OK
2,ser_geoportal_bandas_aparcamiento,data/interim/cartografia/ser_geoportal_bandas_aparcamiento/ser_geoportal_ban...,"(34450, 4)",25830,1.273,OK,OK,OK


### Lectura final

Las tres capas Geoportal reproducen los esquemas, dimensiones, atributos, CRS y geometrías de las salidas de referencia. El límite SER queda disponible como geometría oficial del ámbito regulado; los barrios SER conservan los 66 polígonos útiles y las dos piezas territoriales asociadas al código 904; y las bandas mantienen las 34.450 líneas reguladas válidas por color, plazas y geometría.

La escritura genera exclusivamente tres Parquet en `data/interim/cartografia/`, con lectura posterior satisfactoria. No se han construido agregados, joins, mapas ni métricas de dificultad.

Con estas capas cerradas, el siguiente bloque del notebook podrá diagnosticar y limpiar `callejero_viales_vigentes`. La discrepancia observada en el recuento de geometrías inválidas del callejero deberá reproducirse y explicarse antes de decidir cualquier reparación o exclusión.